# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# A `.env` file should exist with the following variables:
#   OPENAI_API_KEY="YOUR_KEY"
#   CHROMA_OPENAI_API_KEY="YOUR_KEY"
#   TAVILY_API_KEY="YOUR_KEY"
# (Optional, when using a Udacity / vocareum proxy)
#   OPENAI_BASE_URL="https://openai.vocareum.com/v1"

In [4]:
# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY", OPENAI_API_KEY)
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")  # may be None for the public OpenAI API

assert OPENAI_API_KEY, "OPENAI_API_KEY is not set. Please populate your .env file."

### VectorDB Instance

In [5]:
# Instantiate the ChromaDB persistent client.
# Storage will live in the local folder `chromadb/`, alongside this notebook.
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# Pick an embedding function. We use OpenAI's `text-embedding-3-small`, which is
# fast, cheap, and high-quality. Make sure the SAME embedding function is used
# whenever the collection is loaded later (e.g. in Part 2).
embedding_kwargs = {
    "api_key": CHROMA_OPENAI_API_KEY,
    "model_name": "text-embedding-3-small",
}
if OPENAI_BASE_URL:
    embedding_kwargs["api_base"] = OPENAI_BASE_URL

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(**embedding_kwargs)

In [7]:
# Create (or load) the collection. Using `get_or_create_collection` makes the
# notebook idempotent: re-running it will not raise if the collection exists.
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)
print(f"Collection ready: {collection.name} (current size: {collection.count()})")

Collection ready: udaplay (current size: 0)


### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

# Skip docs already indexed (idempotent re-runs)
existing_ids = set(collection.get().get("ids", []))

added = 0
for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Use the file stem (e.g. "001") as a stable document id
    doc_id = os.path.splitext(file_name)[0]
    if doc_id in existing_ids:
        continue

    # Compose a rich text representation that the embedder can index. Including
    # all key fields gives semantic search a stronger signal than just the
    # description alone.
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
        f"Genre: {game.get('Genre', 'N/A')}. "
        f"Publisher: {game.get('Publisher', 'N/A')}. "
        f"{game.get('Description', '')}"
    )

    # ChromaDB metadata values must be primitives (str/int/float/bool).
    metadata = {k: v for k, v in game.items() if isinstance(v, (str, int, float, bool))}

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[metadata],
    )
    added += 1

print(f"Indexed {added} new game(s). Collection now has {collection.count()} documents.")

Indexed 15 new game(s). Collection now has 15 documents.


### Sanity check
Run a quick semantic query to confirm everything is wired up correctly.

In [9]:
results = collection.query(
    query_texts=["realistic racing simulator on PlayStation"],
    n_results=3,
)
for doc, meta, dist in zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0]
):
    print(f"- ({dist:.4f}) {meta.get('Name')} [{meta.get('Platform')}, {meta.get('YearOfRelease')}]")
    print(f"  {doc}\n")

- (0.3628) Gran Turismo [PlayStation 1, 1997]
  [PlayStation 1] Gran Turismo (1997) - Genre: Racing. Publisher: Sony Computer Entertainment. A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.

- (0.3894) Gran Turismo 5 [PlayStation 3, 2010]
  [PlayStation 3] Gran Turismo 5 (2010) - Genre: Racing. Publisher: Sony Computer Entertainment. A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.

- (0.5994) Grand Theft Auto: San Andreas [PlayStation 2, 2004]
  [PlayStation 2] Grand Theft Auto: San Andreas (2004) - Genre: Action-adventure. Publisher: Rockstar Games. An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.

